In [1]:
import pandas as pd




In [2]:
df=pd.read_csv(r'c:\\Users\\HP\\Downloads\\Dataset_digital_life_impact_on_habits\\social_media_sleep_stress_productivity_11000.csv')
df.nunique()



user_id                    11000
age                           34
daily_screen_time_hours     1101
social_media_hours           751
sleep_hours                  601
exercise_minutes             120
study_work_hours             901
productivity_score          6633
stress_level                   3
platform                       5
dtype: int64

In [3]:
df["age"].unique()

array([44, 30, 23, 36, 34, 38, 26, 39, 18, 37, 17, 45, 48, 27, 40, 42, 43,
       31, 22, 24, 33, 19, 29, 41, 35, 32, 21, 49, 25, 46, 16, 20, 28, 47])

## Data Dictionary

A data dictionary describes each variable in the dataset, its meaning,
its storage datatype, and its statistical data type. This helps determine
which analysis techniques are appropriate for each variable.
| Variable | Pandas dtype | Statistical Type | Role / Meaning |
|---|---|---|---|
| `user_id` | int64 | Identifier | Uniquely identifies a user/record |
| `age` | int64 | Numerical - Discrete | Age of the user |
| `daily_screen_time_hours` | float64 | Numerical - Continuous | Daily screen time in hours |
| `social_media_hours` | float64 | Numerical - Continuous | Daily social media usage in hours |
| `sleep_hours` | float64 | Numerical - Continuous | Sleep duration in hours |
| `exercise_minutes` | int64 | Numerical - Discrete | Exercise duration in minutes |
| `study_work_hours` | float64 | Numerical - Continuous | Daily study/work duration |
| `productivity_score` | float64 | Numerical - Continuous | Productivity score |
| `stress_level` | object | Categorical - Ordinal | Stress category: Low, Medium, High |
| `platform` | object | Categorical - Nominal | Social media platform |

In [4]:
identifier_column = ["user_id"]

numerical_columns = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "sleep_hours",
    "exercise_minutes",
    "study_work_hours",
    "productivity_score"
]

categorical_columns = [
    "stress_level",
    "platform"
]

## Data Quality Assessment

Before performing statistical analysis, the dataset is examined for
potential quality problems such as missing values, duplicate records,
duplicate identifiers, inconsistent categories, and invalid values.

In [5]:
print(df.isnull().sum())
if(df.isnull().sum() > 0).any():
    print("missing values are present in the dataset") 
    #to remove missing values from the dataset
else:
    print("No missing values are present in the dataset")

missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)


user_id                    0
age                        0
daily_screen_time_hours    0
social_media_hours         0
sleep_hours                0
exercise_minutes           0
study_work_hours           0
productivity_score         0
stress_level               0
platform                   0
dtype: int64
No missing values are present in the dataset
user_id                    0.0
age                        0.0
daily_screen_time_hours    0.0
social_media_hours         0.0
sleep_hours                0.0
exercise_minutes           0.0
study_work_hours           0.0
productivity_score         0.0
stress_level               0.0
platform                   0.0
dtype: float64


In [6]:
df.duplicated().sum()
if(df.duplicated().sum() > 0):
    print("Duplicate values are present in the dataset")
    #to remove duplicate values from the dataset
    df.drop_duplicates(inplace=True)
else:
    print("No duplicate values are present in the dataset")

#if they are duplicate values in the datset , how to handle them 

No duplicate values are present in the dataset


### Duplicate Records

No completely duplicated records were found in the dataset. Therefore,
no rows need to be removed on the basis of complete duplication.

In [7]:
constraints = {
    "age": (0, 100),
    "daily_screen_time_hours": (0, 24),
    "social_media_hours": (0, 24),
    "sleep_hours": (0, 24),
    "exercise_minutes": (0, 1440),
    "study_work_hours": (0, 24),
    "productivity_score": (0, 100)
}

# Create validation DataFrame
validation = pd.DataFrame(index=df.index)

# Step 2: Check numerical constraints
for column, (minimum, maximum) in constraints.items():
    validation[column] = df[column].between(minimum, maximum) #give boolean values
    #if not exist in the range gives False

# Identify rows where any constraint fails
invalid_condition = ~validation.all(axis=1) #reverse the booelan values and gives True for the invalide rows

# Step 4: Display original records that failed validation
invalid_records = df[invalid_condition]

display(invalid_records)

# Step 5: Count records requiring investigation
print("Invalid percentage:", len(invalid_records)/len(df)*100,"%")


,user_id,age,daily_screen_time_hours,social_media_hours,sleep_hours,exercise_minutes,study_work_hours,productivity_score,stress_level,platform


Invalid percentage: 0.0 %


In [8]:
# Identify potentially inconsistent records

condition = df["social_media_hours"] > df["daily_screen_time_hours"]
logical_inconsistencies = df[condition]

print("Potentially inconsistent records:", len(logical_inconsistencies))
logical_inconsistencies.head()

Potentially inconsistent records: 3217


,user_id,age,daily_screen_time_hours,social_media_hours,sleep_hours,exercise_minutes,study_work_hours,productivity_score,stress_level,platform
2,3,23,2.56,4.64,4.69,8,4.47,60.07,High,YouTube
6,7,26,6.37,6.72,3.49,10,2.03,6.58,Low,LinkedIn
8,9,39,6.73,6.79,7.47,102,1.37,9.88,High,LinkedIn
12,13,17,2.18,7.53,7.51,86,9.31,82.97,Low,YouTube
24,25,30,1.02,3.92,4.40,59,5.52,36.60,Medium,Instagram


In [9]:
df["stress_level"].value_counts(dropna=True)  #count distinct value with NaN


stress_level
Low       3715
Medium    3681
High      3604
Name: count, dtype: int64

In [10]:
df["platform"].value_counts(dropna=True)

platform
LinkedIn     2239
Snapchat     2235
YouTube      2200
Twitter      2183
Instagram    2143
Name: count, dtype: int64

In [11]:
df_analysis = df.copy()

df_analysis["logical_consistency"] = (
    df_analysis["social_media_hours"]
    <= df_analysis["daily_screen_time_hours"]
)

## Data Quality Assessment Summary

The dataset was examined to identify potential quality issues before further analysis.

The following checks were performed:

1. **Missing Values:** Checked for null values using `isnull().sum()`.
2. **Duplicate Records:** Checked for completely duplicated rows using `duplicated().sum()`.
3. **Duplicate User IDs:** Checked whether user identifiers were repeated.
4. **Numerical Validity:** Examined minimum and maximum values of numerical variables.
5. **Logical Consistency:** Investigated whether social media usage exceeded total daily screen time.
6. **Categorical Consistency:** Examined the categories and frequencies of `platform` and `stress_level`.

These checks help assess the dataset's completeness, uniqueness, validity, and consistency.

### Data Quality Assessment — Findings

The dataset contains **11,000 records and 10 variables**. Data quality was assessed by examining completeness, duplication, numerical validity, categorical consistency, and logical consistency.

| Quality Dimension | Findings |
|---|---|
| Missing values | No missing values detected |
| Duplicate records | No completely duplicated records detected |
| Numerical validity | No violations reported under the defined numerical constraints |
| Categorical consistency | No inconsistent category labels reported |
| Logical consistency | 3,128 records flagged where social media usage exceeds daily screen time |

**Key Finding**

The dataset is complete and contains no completely duplicated records. Numerical values satisfy the proposed range constraints, and no categorical inconsistencies were reported.

However, 3,128 records were identified where `social_media_hours` exceeds `daily_screen_time_hours`. These records represent a potential logical inconsistency, assuming social media usage is included within total daily screen time.

The flagged records were retained rather than deleted or modified because the correct values cannot be established from the available information.

**Conclusion**

The dataset passed the reported completeness, duplicate-record, numerical-range, and categorical-consistency checks. The identified logical inconsistency remains a limitation that must be considered during further analysis.